In [32]:
from typing import Final
import numpy as np
import pandas as pd


DATA_FILENAME: Final[str] = "FCM_Dev Trader_Case.xlsx"

In [2]:
def get_universe() -> pd.DataFrame:
    """Get coverage universe and some static data."""
    universe = pd.read_excel(
        DATA_FILENAME,
        sheet_name="Coverage Universe",
        usecols="B:M",
        index_col=0,
        skiprows=1,
    )
    return universe


universe = get_universe()
universe.head()

,BBG Ticker to Fetch Info,BBG or RT,Analyst,Ticker_Original,Cusip,SEDOL,Ticker and Exchange Code,Exchange Code,Company Long Name,IPO Date,Most Recent Trading Day
Fernbridge Ticker,,,,,,,,,,,
4324.TYS,4324 JP,RT,Analyst2,NaN,NaN,6416281,4324 JP,JP,Dentsu Group Inc,#N/A Invalid Security,2/27/2025
6758.TYS,6758 JP,RT,Analyst2,NaN,NaN,6821506,6758 JP,JP,Sony Group Corp,#N/A Invalid Security,2/27/2025
7974.TYS,7974 JP,RT,Analyst2,NaN,NaN,6639550,7974 JP,JP,Nintendo Co Ltd,#N/A Invalid Security,2/27/2025
ABB,ABBN SW,BBG,Analyst3,NaN,NaN,7108899,ABBN SW,SW,ABB Ltd,NaN,2/27/2025
ABNB,ABNB,BBG,Analyst2,NaN,009066101,BMGYYH4,ABNB US,US,Airbnb Inc,12/10/2020,2/27/2025


In [16]:
def get_mapping_from_bbg_to_fernbridge_ticker() -> pd.Series:
    return pd.Series(
        {
            bbg_ticker: fernbridge_ticker
            for fernbridge_ticker, bbg_ticker
            in universe["BBG Ticker to Fetch Info"].items()
        }
    )


mapping_from_bbg_to_fernbridge_ticker = get_mapping_from_bbg_to_fernbridge_ticker()
mapping_from_bbg_to_fernbridge_ticker

4324 JP    4324.TYS
6758 JP    6758.TYS
7974 JP    7974.TYS
ABBN SW         ABB
ABNB           ABNB
             ...   
YUM             YUM
ZEN             ZEN
ZI               ZI
ZM               ZM
ZS               ZS
Length: 297, dtype: object

In [13]:
def get_excess_returns_for_index(
    # TODO(sparsh): Not provided, but could look up actual financing rates
    #     based on e.g. 3M TBill + assumed spread.
    financing_rate: pd.Series | float = 0.0,
) -> pd.DataFrame:
    """Get excess returns for indices, provided a financing rate."""
    px = pd.read_excel(
        DATA_FILENAME,
        sheet_name="Raw Market Data",
        usecols="A:D",
        index_col=0,
        skiprows=2,
    )
    px = px.asfreq(freq="D")
    px = px.ffill()
    r = px.shift() / px - 1
    # Get rid of spurious zero returns over weekends.
    # TODO(sparsh): Take into account trading holidays
    #     using e.g. `holidays` or `pandas-market-calendars`.
    r = r.asfreq(freq="B")
    xr = r - financing_rate
    return xr


excess_returns_for_index = get_excess_returns_for_index()
excess_returns_for_index.head()

,SPXT,XNDX,XCMP
Date,,,
2024-01-01,0.000000,0.000000,0.000000
2024-01-02,0.005661,0.016949,0.016570
2024-01-03,0.008039,0.010588,0.011838
2024-01-04,0.003308,0.005282,0.005620
2024-01-05,-0.001823,-0.001470,-0.000952


In [30]:
def get_excess_returns_for_stocks(
    # TODO(sparsh): Not provided, but could look up actual financing rates
    #     based on e.g. 3M TBill + assumed spread.
    financing_rate: pd.Series | float = 0.0,
) -> pd.DataFrame:
    """Get excess returns for stocks, provided a financing rate."""
    r = pd.read_excel(
        DATA_FILENAME,
        sheet_name="Raw Market Data",
        usecols="H,O:KY",
        index_col=0,
        skiprows=2,
    )
    # drop blank first row
    r = r.iloc[1:, :]
    r = r.asfreq(freq="D")
    r = r.fillna(0)
    # Get rid of spurious zero returns over weekends.
    # TODO(sparsh): Take into account trading holidays
    #     using e.g. `holidays` or `pandas-market-calendars`.
    r = r.asfreq(freq="B")
    xr = r - financing_rate
    # rename
    unmapped_tickers = xr.columns.difference(
        mapping_from_bbg_to_fernbridge_ticker.index
    )
    if not unmapped_tickers.empty:
        raise RuntimeError(f"Unmapped tickers! {unmapped_tickers}")
    xr = xr.rename(columns=mapping_from_bbg_to_fernbridge_ticker)
    xr = xr.loc[:, universe.index]
    return xr


excess_returns_for_stocks = get_excess_returns_for_stocks()
excess_returns_for_stocks.head()

Fernbridge Ticker,4324.TYS,6758.TYS,7974.TYS,ABB,ABNB,AC.PAR,ACN,ADBE,ADI,ADSK,...,WORK,WPP.LSE,WU,WYNN,YELP,YUM,ZEN,ZI,ZM,ZS
Start,,,,,,,,,,,,,,,,,,,,,
2024-01-01,0.00000,0.000000,0.000000,0.000000,-0.012193,0.001156,-0.011370,-0.027707,-0.025081,-0.038443,...,0.0,-0.009562,0.017617,0.038305,-0.020490,-0.012475,0.0,-0.038940,-0.038381,-0.041479
2024-01-02,0.00000,0.000000,0.000000,-0.031635,-0.007882,-0.007794,-0.025943,-0.014274,-0.023866,-0.029600,...,0.0,-0.021990,-0.024732,-0.006025,-0.007764,0.000543,0.0,-0.051210,-0.028633,-0.010030
2024-01-03,0.02377,-0.023490,-0.024868,0.009690,0.002249,0.004364,-0.002456,-0.008290,-0.015294,0.007615,...,0.0,0.017549,-0.006762,0.001914,0.002825,-0.003176,0.0,-0.002372,-0.004020,0.000428
2024-01-04,0.00432,0.005727,0.006550,-0.013436,0.016901,-0.007242,-0.001394,-0.004321,0.002580,0.002621,...,0.0,0.037726,0.017872,0.015285,-0.025358,-0.002720,0.0,0.014269,0.000897,-0.002472
2024-01-05,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000


In [25]:
def _get_pflio_dta() -> pd.DataFrame:
    """Pull info from "Portfolio" tab."""
    dta = pd.read_excel(
        DATA_FILENAME,
        sheet_name="Portfolio",
        usecols="B:K",
        index_col=[0, 2, 3],
        parse_dates=[3],
        skiprows=2,
    )
    dta = dta.sort_index()
    return dta


_pflio_dta = _get_pflio_dta()
_pflio_dta.head()

Label  \
Level (Fund / Analyst / Stock) Adjusted Label Date                   
Analyst                        Analyst1       2024-01-01  Analyst1   
                                              2024-01-02  Analyst1   
                                              2024-01-03  Analyst1   
                                              2024-01-04  Analyst1   
                                              2024-01-05  Analyst1   

                                                          Combined_G/L_Daily  \
Level (Fund / Analyst / Stock) Adjusted Label Date                             
Analyst                        Analyst1       2024-01-01        0.000000e+00   
                                              2024-01-02       -5.015726e+06   
                                              2024-01-03       -1.717630e+06   
                                              2024-01-04       -4.679475e+05   
                                              2024-01-05       -6.096062e+05   

                                                          Combined_Exposure_EOD  \
Level (Fund / Analyst / Stock) Adjusted Label Date                                
Analyst                        Analyst1       2024-01-01           1.787261e+08   
                                              2024-01-02           1.787261e+08   
                                              2024-01-03           1.720464e+08   
                                              2024-01-04           1.698800e+08   
                                              2024-01-05           1.682185e+08   

                                                          Return_Daily  \
Level (Fund / Analyst / Stock) Adjusted Label Date                       
Analyst                        Analyst1       2024-01-01      0.000000   
                                              2024-01-02     -0.028064   
                                              2024-01-03     -0.009984   
                                              2024-01-04     -0.002755   
                                              2024-01-05     -0.003624   

                                                          Raw Stock Return_Daily  \
Level (Fund / Analyst / Stock) Adjusted Label Date                                 
Analyst                        Analyst1       2024-01-01                     NaN   
                                              2024-01-02                     NaN   
                                              2024-01-03                     NaN   
                                              2024-01-04                     NaN   
                                              2024-01-05                     NaN   

                                                          Index Return_Daily  \
Level (Fund / Analyst / Stock) Adjusted Label Date                             
Analyst                        Analyst1       2024-01-01            0.000000   
                                              2024-01-02           -0.005629   
                                              2024-01-03           -0.007974   
                                              2024-01-04           -0.003298   
                                              2024-01-05            0.001828   

                                                          Contribution_Daily  
Level (Fund / Analyst / Stock) Adjusted Label Date                            
Analyst                        Analyst1       2024-01-01            0.000000  
                                              2024-01-02           -0.005828  
                                              2024-01-03           -0.002000  
                                              2024-01-04           -0.000539  
                                              2024-01-05           -0.000717

In [29]:
def get_fund_nav() -> pd.Series:
    """Get fund-level NAV by day."""
    nav = _pflio_dta.loc[("Fund", "Fund Total"), "Combined_Exposure_EOD"]
    nav = nav.asfreq("D")
    nav = nav.ffill()
    nav = nav.asfreq("B")
    return nav


fund_nav = get_fund_nav()
fund_nav.head()

Date
2024-01-01    8.606660e+08
2024-01-02    8.606742e+08
2024-01-03    8.588705e+08
2024-01-04    8.684963e+08
2024-01-05    8.508014e+08
Freq: B, Name: Combined_Exposure_EOD, dtype: float64

In [58]:
def get_pflio_wts() -> pd.DataFrame:
    """Get portfolio weights by day."""
    dta = _pflio_dta.loc["Stock", :]
    # sign
    true_sign_of_exposure = np.sign(dta["Combined_G/L_Daily"])
    unsigned_exposure = dta["Combined_Exposure_EOD"]
    recorded_sign_of_exposure = np.sign(unsigned_exposure)
    assert not (recorded_sign_of_exposure < 0).any(), "Assumed sign convention is wrong!"
    signed_exposure = true_sign_of_exposure * unsigned_exposure
    assert isinstance(signed_exposure, pd.Series), type(signed_exposure)
    signed_exposure = signed_exposure.unstack(level="Adjusted Label")
    signed_exposure = signed_exposure.asfreq(freq="D")
    signed_exposure = (
        signed_exposure
        .ffill()
        # if we've never held this position, propagate zero
        .fillna(0)
    )
    signed_exposure = signed_exposure.asfreq("B")
    wts = signed_exposure.divide(fund_nav, axis="index")
    unrecognized_tickers = wts.columns.difference(universe.index)
    if not unrecognized_tickers.empty:
        raise RuntimeError(unrecognized_tickers)
    wts = wts.reindex(columns=universe.index).fillna(0)
    return wts


pflio_wts = get_pflio_wts()
pflio_wts.head()

Fernbridge Ticker,4324.TYS,6758.TYS,7974.TYS,ABB,ABNB,AC.PAR,ACN,ADBE,ADI,ADSK,...,WORK,WPP.LSE,WU,WYNN,YELP,YUM,ZEN,ZI,ZM,ZS
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [62]:
(pflio_wts != 0).mean().mean()

0.08981674248849822